# 04 Graphene Reference Alignment

This notebook is an opt-in Graphene reference check. It uses dftbephy only as an external physics/numerical reference and keeps DeePTB's API and NPZ workflow as the supported contract. It reads the dftbephy Graphene HDF5 reference but does not write dftbephy HDF5, HSD, CLI input, or output directories.


In [ ]:
from pathlib import Path
import json
import os
import sys

import h5py
import numpy as np
from scipy import linalg

from dptb.postprocess.unified.eph.contraction import compute_coupling_matrix
from dptb.postprocess.unified.eph import providers as eph_providers
from dptb.postprocess.unified.eph.providers import _primitive_to_supercell_from_supercell_to_primitive
from dptb.postprocess.unified.eph.utils import reshape_phonopy_eigenvectors

reference_root_env = os.environ.get("DEEPTB_EPH_REFERENCE_ROOT")
if not reference_root_env:
    raise FileNotFoundError("Set DEEPTB_EPH_REFERENCE_ROOT to the external dftbephy checkout before running this notebook.")
REFERENCE_ROOT = Path(reference_root_env).expanduser()
if not REFERENCE_ROOT.exists():
    raise FileNotFoundError(f"DEEPTB_EPH_REFERENCE_ROOT does not exist: {REFERENCE_ROOT}")
GRAPHENE = REFERENCE_ROOT / "examples" / "Graphene"
WORK = Path.cwd() / "work"
WORK.mkdir(exist_ok=True)

required = [
    GRAPHENE / "phonons" / "phonopy_disp.yaml",
    GRAPHENE / "phonons" / "FORCE_SETS",
    GRAPHENE / "el-ph" / "reference.npz",
    GRAPHENE / "el-ph" / "derivatives.npz",
    GRAPHENE / "el-ph" / "el-ph-Nq4-K-bandsel.hdf5",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing external Graphene reference files:\n" + "\n".join(missing))

if str(REFERENCE_ROOT) not in sys.path:
    sys.path.insert(0, str(REFERENCE_ROOT))

import phonopy
import dftbephy.dftb as dftbephy_dftb
import dftbephy.fourier as dftbephy_fourier
import dftbephy.units as dftbephy_units


## Load reference data

The reference arrays are used only to check the DeePTB contraction convention: electronic eigenvalues at `k`, eigenvalues at `k+q`, and `g2` / `coupling_strength`.


In [ ]:
phonons = phonopy.load(
    str(GRAPHENE / "phonons" / "phonopy_disp.yaml"),
    force_sets_filename=str(GRAPHENE / "phonons" / "FORCE_SETS"),
)
primitive = phonons.primitive
supercell = phonons.supercell
num_supercells = int(round(abs(np.linalg.det(supercell.supercell_matrix))))
num_primitive_atoms = len(primitive)

supercell_atom_to_cell = np.tile(np.arange(num_supercells), num_primitive_atoms)
primitive_map = supercell.u2u_map
supercell_to_primitive_atom = np.array([primitive_map[atom] for atom in supercell.s2u_map], dtype=int)
primitive_to_supercell_atom = _primitive_to_supercell_from_supercell_to_primitive(
    supercell_to_primitive_atom,
    num_primitive_atoms,
    supercell_atom_to_cell=supercell_atom_to_cell,
    preferred_cell=(num_supercells - 1) // 2,
)

angular_momenta = {"C": ["s", "p"]}
supercell_orbitals = [
    sum([dftbephy_dftb.std_orbital_order[angular] for angular in angular_momenta[symbol]], [])
    for symbol in supercell.symbols
]
supercell_orbital_offsets = np.insert(np.cumsum([len(orbs) for orbs in supercell_orbitals]), 0, 0)
primitive_orbitals = [supercell_orbitals[index] for index in primitive_to_supercell_atom]
primitive_orbital_offsets = np.insert(np.cumsum([len(orbs) for orbs in primitive_orbitals]), 0, 0)
orbital_slices = [
    (int(primitive_orbital_offsets[index]), int(primitive_orbital_offsets[index + 1]))
    for index in range(num_primitive_atoms)
]
shortest_vectors, vector_multiplicity = eph_providers._phonopy_supercell_maps(phonons)[3:]

with np.load(GRAPHENE / "el-ph" / "reference.npz", allow_pickle=False) as data:
    reference_hamiltonian = data["H0"]
    reference_overlap = data["S0"]
with np.load(GRAPHENE / "el-ph" / "derivatives.npz", allow_pickle=False) as data:
    h_derivatives_supercell = data["H_derivs"]
    overlap_derivatives_supercell = data["S_derivs"]
with h5py.File(GRAPHENE / "el-ph" / "el-ph-Nq4-K-bandsel.hdf5", "r") as data:
    qpoints = data["ph/qpoints"][()]
    frequencies = data["ph/omega"][()] / dftbephy_units.THZ__EV
    reference_strength = data["el-ph/g2_0"][()]
    kpoint = data["el-ph/g2_0"].attrs["kvec"]
    reference_eigenvalues_k = data["el/eps_0"][()]
    reference_eigenvalues_kq = data["el/eps_q_0"][()]

phonons.run_qpoints(qpoints, with_eigenvectors=True)
phonon_eigenvectors = reshape_phonopy_eigenvectors(
    phonons.get_qpoints_dict()["eigenvectors"],
    num_primitive_atoms,
)
print("qpoints", qpoints.shape)
print("reference g2", reference_strength.shape)


## Recompute DeePTB contraction inputs

This cell uses dftbephy Fourier helpers only to reconstruct the same benchmark matrices. The EPC contraction and phase/mass/frequency convention under check is DeePTB's `compute_coupling_matrix(...)`.


In [ ]:
def lattice_ft(matrix, kvec):
    return dftbephy_fourier.calculate_lattice_ft(
        matrix,
        kvec,
        primitive_to_supercell_atom,
        supercell_to_primitive_atom,
        supercell_atom_to_cell,
        primitive_orbital_offsets,
        supercell_orbital_offsets,
        shortest_vectors,
        vector_multiplicity,
    )


def lattice_ft_derivative(derivatives, kvec):
    return dftbephy_fourier.calculate_lattice_ft_derivative(
        derivatives,
        kvec,
        primitive_to_supercell_atom,
        supercell_to_primitive_atom,
        supercell_atom_to_cell,
        primitive_orbital_offsets,
        supercell_orbital_offsets,
        shortest_vectors,
        vector_multiplicity,
    )

hamiltonian_k = lattice_ft(reference_hamiltonian, kpoint)
overlap_k = lattice_ft(reference_overlap, kpoint)
eigenvalues_k, eigenvectors_k = linalg.eigh(hamiltonian_k, b=overlap_k)
h_derivatives_k = lattice_ft_derivative(h_derivatives_supercell, kpoint)[None]
overlap_derivatives_k = lattice_ft_derivative(overlap_derivatives_supercell, kpoint)[None]

eigenvalues_kq = []
eigenvectors_kq = []
h_derivatives_kq = []
overlap_derivatives_kq = []
for qpoint in qpoints:
    kqpoint = kpoint + qpoint
    hamiltonian_kq = lattice_ft(reference_hamiltonian, kqpoint)
    overlap_kq = lattice_ft(reference_overlap, kqpoint)
    bands_kq, states_kq = linalg.eigh(hamiltonian_kq, b=overlap_kq)
    eigenvalues_kq.append(bands_kq)
    eigenvectors_kq.append(states_kq)
    h_derivatives_kq.append(lattice_ft_derivative(h_derivatives_supercell, kqpoint))
    overlap_derivatives_kq.append(lattice_ft_derivative(overlap_derivatives_supercell, kqpoint))


## Compare against dftbephy / paper Graphene reference

The tolerances match the opt-in pytest benchmark. A passing cell means the DeePTB contraction reproduces the reference eigenvalues and `g2` values for this Graphene fixture.


In [ ]:
_, coupling_strength = compute_coupling_matrix(
    eigenvalues_k=eigenvalues_k[None],
    eigenvectors_k=eigenvectors_k[None],
    eigenvalues_kq=np.asarray(eigenvalues_kq)[:, None, :],
    eigenvectors_kq=np.asarray(eigenvectors_kq)[:, None, :, :],
    h_derivatives_k=h_derivatives_k,
    h_derivatives_kq=np.asarray(h_derivatives_kq)[:, None, :, :, :],
    overlap_derivatives_k=overlap_derivatives_k,
    overlap_derivatives_kq=np.asarray(overlap_derivatives_kq)[:, None, :, :, :],
    phonon_eigenvectors=phonon_eigenvectors,
    masses=np.asarray(supercell.masses)[primitive_to_supercell_atom],
    frequencies=frequencies,
    qpoints=qpoints,
    scaled_positions=primitive.scaled_positions,
    orbital_slices=orbital_slices,
    derivative_mode="row",
    prefactor_amu_thz=dftbephy_units.hbar_amu_THz,
)

metrics = {
    "max_abs_eigenvalues_k_error_ev": float(np.max(np.abs(eigenvalues_k - reference_eigenvalues_k))),
    "max_abs_eigenvalues_kq_error_ev": float(np.max(np.abs(np.asarray(eigenvalues_kq) - reference_eigenvalues_kq))),
    "max_abs_g2_error_ev2": float(np.max(np.abs(coupling_strength[:, 0] - reference_strength))),
    "reference_shape": list(reference_strength.shape),
    "deeptb_shape": list(coupling_strength[:, 0].shape),
}

np.testing.assert_allclose(eigenvalues_k, reference_eigenvalues_k, atol=1e-12, rtol=0.0)
np.testing.assert_allclose(np.asarray(eigenvalues_kq), reference_eigenvalues_kq, atol=1e-12, rtol=0.0)
np.testing.assert_allclose(coupling_strength[:, 0], reference_strength, atol=5e-12, rtol=1e-12)

(WORK / "graphene_reference_alignment.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
